In [297]:
#%pip install pandas openpyxl

In [298]:
#%pip install openpyxl

In [299]:
#%pip install seaborn matplotlib

In [300]:
import pandas as pd
import openpyxl
import seaborn

In [301]:
df_creditos_saude = pd.read_excel("Base_Case_Credito_Profissionais_Saude.xlsx", sheet_name="base_inadimplencia")

## **Visão Geral da tabela**
### conhecendo as colunas e valores

In [302]:
df_creditos_saude.info()

<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Profissao           184 non-null    str  
 1   Tempo_Atuacao_Anos  184 non-null    str  
 2   Faixa_Renda_Mensal  184 non-null    str  
 3   Tipo_Produto        184 non-null    str  
 4   Valor_Contrato      184 non-null    int64
 5   Inadimplente        184 non-null    bool 
 6   Inadimplente_FLAG   184 non-null    int64
dtypes: bool(1), int64(2), str(4)
memory usage: 8.9 KB


In [303]:
for coluna in df_creditos_saude.columns:
    if coluna == "Valor_Contrato":
        break
    else:
        print(coluna)
        print(df_creditos_saude[coluna].unique())
        print()

Profissao
<StringArray>
['Dentista', 'Enfermeiro', 'Fisioterapeuta', 'Medico', 'Outros']
Length: 5, dtype: str

Tempo_Atuacao_Anos
<StringArray>
['0-2', '6-10', '10+', '3-5']
Length: 4, dtype: str

Faixa_Renda_Mensal
<StringArray>
['Até 5k', '10k-20k', '5k-10k', '20k+']
Length: 4, dtype: str

Tipo_Produto
<StringArray>
['Credito Pessoal', 'Capital de Giro', 'Equipamentos']
Length: 3, dtype: str



In [304]:
df_creditos_saude["Faixa_Renda_Mensal"] = df_creditos_saude["Faixa_Renda_Mensal"].replace("5k-10k", "até 10k")
df_creditos_saude["Faixa_Renda_Mensal"] = df_creditos_saude["Faixa_Renda_Mensal"].replace("10k-20k", "até 20k")


In [305]:
for coluna in df_creditos_saude.columns:
    if coluna == "Valor_Contrato":
        break
    else:
        print(coluna)
        print(df_creditos_saude[coluna].unique())
        print()

Profissao
<StringArray>
['Dentista', 'Enfermeiro', 'Fisioterapeuta', 'Medico', 'Outros']
Length: 5, dtype: str

Tempo_Atuacao_Anos
<StringArray>
['0-2', '6-10', '10+', '3-5']
Length: 4, dtype: str

Faixa_Renda_Mensal
<StringArray>
['Até 5k', 'até 20k', 'até 10k', '20k+']
Length: 4, dtype: str

Tipo_Produto
<StringArray>
['Credito Pessoal', 'Capital de Giro', 'Equipamentos']
Length: 3, dtype: str



In [306]:
df_creditos_saude.describe()

,Valor_Contrato,Inadimplente_FLAG
count,184.000000,184.0
mean,50910.586957,1.0
std,21348.526704,0.0
min,5000.000000,1.0
25%,35829.250000,1.0
50%,50059.000000,1.0
75%,68058.500000,1.0
max,101976.000000,1.0


In [307]:
df_creditos_saude["Valor_Contrato"].min()

np.int64(5000)

In [308]:
df_creditos_saude["Valor_Contrato"].max()

np.int64(101976)

In [309]:
df_creditos_saude["Valor_Contrato"].mean()

np.float64(50910.586956521736)

In [310]:
df_creditos_ = df_creditos_saude.copy()
df_creditos_ = df_creditos_.reset_index(drop=True)

In [311]:
for indice_linha in range(len(df_creditos_)):
    registro_selecionado = df_creditos_.loc[indice_linha, "Tipo_Produto"]
    if registro_selecionado == "Capital de Giro":
        df_creditos_.at[indice_linha, "PJ"] = True
    else:
        df_creditos_.at[indice_linha, "PJ"] = False

C:\Users\49582707879\AppData\Local\Temp\ipykernel_26248\709954270.py:6: Pandas4Warning: Setting a value on a DataFrame via .at with a key that does not exist in the index is deprecated and will raise a KeyError in a future version. Use .loc instead.
  df_creditos_.at[indice_linha, "PJ"] = False


### Entre registros filtrados inadimplência, apenas 184 deles correspondem como verdadeiro, ou seja 9.2% de 2000 registros

In [312]:
df_inadimplencia_ = df_creditos_[df_creditos_["Inadimplente"] == True]
df_inadimplencia_ = df_inadimplencia_.reset_index(drop=True)
print("Quantidade de Profissionais Inadimplentes:", len(df_inadimplencia_))
print(f"Porcentagem de inadimplêntes: {round((len(df_inadimplencia_)* 100)/ len(df_creditos_),2)}%")


Quantidade de Profissionais Inadimplentes: 184
Porcentagem de inadimplêntes: 100.0%


---

In [313]:
total_registros = len(df_inadimplencia_)
print(total_registros)

184


In [314]:
print((len(df_inadimplencia_)/len(df_creditos_))*100)

100.0


### Para os 184 valores presentes em inadimplência:
- É possível concluir que o tipo de contrato mais possível corresponde a valores solicitados para **Crédito Pessoal**
- Enquanto **Capital de Giro** corresponde a menos de 1/5 

| Crédito Pessoal | Equipamentos | Capital de Giro |
|:---------------:|:------------:|:---------------:|
| 86              | 63           | 35              |
| 46,74%          | 34,24%       | 19,02%          |

In [315]:
lista_tipos = ["Credito Pessoal", "Equipamentos", "Capital de Giro"]
for tipo_produto in lista_tipos:
    contador_tipo = 0
    for linha_contador in range(len(df_inadimplencia_)):
        registro_contador = df_inadimplencia_.loc[linha_contador, "Tipo_Produto"]
        if tipo_produto == registro_contador:
            contador_tipo+=1
    print(tipo_produto)
    print("Quantidade de registros:", contador_tipo)
    print(f"Valor em porcentagem: {round((contador_tipo * 100) / total_registros, 2)}%")
    print()
    

Credito Pessoal
Quantidade de registros: 86
Valor em porcentagem: 46.74%

Equipamentos
Quantidade de registros: 63
Valor em porcentagem: 34.24%

Capital de Giro
Quantidade de registros: 35
Valor em porcentagem: 19.02%



---

### Com base nessa análise, é possível afirmar que:
- a quantidade de inadimplência é bem distribuída entre as profissões, revelando uma diferença mínima entre eles. 
- Apesar disso, Médicos e Dentista tendenciam a ser maioria dentro de inadimplentes.

| Médico          | Dentista     | Fisiote         | Enfermeiro      |  Outros         |
|:---------------:|:------------:|:---------------:|:---------------:|:---------------:|
| 46              | 34           | 33              |   29            |  42             | 
| 25%             | 18,48%       | 17,93%          |   15,76%        |  22,83%         | 


In [316]:
profissoes = ['Dentista', 'Outros', 'Fisioterapeuta', 'Enfermeiro', 'Medico']
for profissao in profissoes:
    contador_prof = 0
    for indice_linha in range(len(df_inadimplencia_)):
        registro = df_inadimplencia_.loc[indice_linha, "Profissao"]
        if profissao == registro:
            contador_prof+=1
    print(profissao)
    print("Quantidade de registros:", contador_prof)
    print(f"Valor em porcentagem: {round((contador_prof * 100)/total_registros,2)}%")
    print()

Dentista
Quantidade de registros: 34
Valor em porcentagem: 18.48%

Outros
Quantidade de registros: 42
Valor em porcentagem: 22.83%

Fisioterapeuta
Quantidade de registros: 33
Valor em porcentagem: 17.93%

Enfermeiro
Quantidade de registros: 29
Valor em porcentagem: 15.76%

Medico
Quantidade de registros: 46
Valor em porcentagem: 25.0%



---

### Fica comprovado que profissionais com **renda mensal** menor que 10k, representam maior número de inadimplentes. 
- É evidênciado que a renda mensal tem impacto direto na inadimplência do profissional.

| Até 5k          | 5k-10k       | 10k-20k         | 20k+            | até 10k |
|:---------------:|:------------:|:---------------:|:---------------:|:---------------:|
| 67              | 60           | 35              |   22            | 127 |
| 36,41%          | 32,61%       | 19,02%          |   11,96%        |69,02%|



In [317]:
renda_mensal = ['Até 5k', 'até 10k', 'até 20k', '20k+']
for renda in renda_mensal:
    contador_renda = 0
    for indice_linha in range(len(df_inadimplencia_)):
        registro = df_inadimplencia_.loc[indice_linha, "Faixa_Renda_Mensal"]
        if renda == registro:
            contador_renda+=1
    print(renda)
    print("Quantidade de registros:", contador_renda)
    print(f"Valor em porcentagem: {round((contador_renda * 100)/ total_registros,2)}%")
    print()

Até 5k
Quantidade de registros: 67
Valor em porcentagem: 36.41%

até 10k
Quantidade de registros: 60
Valor em porcentagem: 32.61%

até 20k
Quantidade de registros: 35
Valor em porcentagem: 19.02%

20k+
Quantidade de registros: 22
Valor em porcentagem: 11.96%



---

## É evidenciado que quanto menor o tempo de atuação profissional, maior é a chance 

In [318]:
anos_atuacao = ['3-5', '0-2', '10+', '6-10']
for tempo_atucao in anos_atuacao:
    contador = 0
    for indice_linha in range(len(df_inadimplencia_)):
        registro = df_inadimplencia_.loc[indice_linha, "Tempo_Atuacao_Anos"]
        if tempo_atucao == registro:
            contador+=1
    print(tempo_atucao)
    print("Quantidade de registros:", contador)
    print(f"Valor em porcentagem: {round((contador * 100)/ total_registros,2)}%")
    print()

3-5
Quantidade de registros: 49
Valor em porcentagem: 26.63%

0-2
Quantidade de registros: 69
Valor em porcentagem: 37.5%

10+
Quantidade de registros: 29
Valor em porcentagem: 15.76%

6-10
Quantidade de registros: 37
Valor em porcentagem: 20.11%



---

### Pessoas que atuem de 5 anos para menos, ocupam mais de 60% dos inadimplentes, mostrando padrão de usuário muito forte em relação a inamdiplência.
- Ainda em cima disso, a quantidade de profissionais que até 2 anos e recebem no máximo 5k, equivalem a 38% em comparação a outros períodos de atuação, ocupanda maior parte dessa faixa  

In [319]:
linhas = []

for tempo_atuacao in anos_atuacao:
    df_temp_profissao = df_inadimplencia_[df_inadimplencia_["Tempo_Atuacao_Anos"] == tempo_atuacao].copy()
    df_temp_profissao = df_temp_profissao.reset_index(drop=True)
    total_registros = len(df_temp_profissao)

    for renda in renda_mensal:
        contador_renda = 0
        for indice_linha in range(len(df_temp_profissao)):
            renda_selecionada = df_temp_profissao.loc[indice_linha, "Faixa_Renda_Mensal"]
            if renda == renda_selecionada:
                contador_renda += 1

        linhas.append({
            "Tempo_Atuacao": tempo_atuacao,
            "Renda": renda,
            "Quant_Registros": contador_renda,
            "%": round((contador_renda * 100) / total_registros, 2),
        })

df_atuacao_renda = pd.DataFrame(linhas)

for temp_atuacao in anos_atuacao:
    soma_reg = df_atuacao_renda[df_atuacao_renda["Tempo_Atuacao"] == temp_atuacao]["Quant_Registros"].sum()
    print(df_atuacao_renda[df_atuacao_renda["Tempo_Atuacao"] == temp_atuacao])
    print("Total de registros:", soma_reg)
    print()

  Tempo_Atuacao    Renda  Quant_Registros      %
0           3-5   Até 5k               18  36.73
1           3-5  até 10k               18  36.73
2           3-5  até 20k                8  16.33
3           3-5     20k+                5  10.20
Total de registros: 49

  Tempo_Atuacao    Renda  Quant_Registros      %
4           0-2   Até 5k               26  37.68
5           0-2  até 10k               22  31.88
6           0-2  até 20k               15  21.74
7           0-2     20k+                6   8.70
Total de registros: 69

   Tempo_Atuacao    Renda  Quant_Registros      %
8            10+   Até 5k               11  37.93
9            10+  até 10k               11  37.93
10           10+  até 20k                4  13.79
11           10+     20k+                3  10.34
Total de registros: 29

   Tempo_Atuacao    Renda  Quant_Registros      %
12          6-10   Até 5k               12  32.43
13          6-10  até 10k                9  24.32
14          6-10  até 20k             

---

In [320]:
df_inadimplencia_ = df_inadimplencia_.drop("Inadimplente_FLAG", axis=1)
df_inadimplencia_ = df_inadimplencia_.sort_values(by=['Profissao', 'Tempo_Atuacao_Anos', 'Tipo_Produto'])
df_inadimplencia_ = df_inadimplencia_.reset_index(drop=True)
df_inadimplencia_

,Profissao,Tempo_Atuacao_Anos,Faixa_Renda_Mensal,Tipo_Produto,Valor_Contrato,Inadimplente,PJ
0,Dentista,0-2,Até 5k,Capital de Giro,37844,True,True
1,Dentista,0-2,até 10k,Capital de Giro,91988,True,True
2,Dentista,0-2,Até 5k,Credito Pessoal,39748,True,False
3,Dentista,0-2,até 20k,Credito Pessoal,55661,True,False
4,Dentista,0-2,Até 5k,Credito Pessoal,67285,True,False
...,...,...,...,...,...,...,...
179,Outros,6-10,até 20k,Credito Pessoal,40839,True,False
180,Outros,6-10,Até 5k,Equipamentos,63818,True,False
181,Outros,6-10,até 10k,Equipamentos,29478,True,False
182,Outros,6-10,até 10k,Equipamentos,73120,True,False


In [321]:
contador = 0
for indice_linha in range(len(df_inadimplencia_)):
    pj_ver = df_inadimplencia_.loc[indice_linha, "PJ"]
    
    if pj_ver:
        contador+=1
print("PJ inadimplentes")
print(f"Quantidade: {contador}")
print(f"Porcentagem: {round(( contador * 100)/ len(df_inadimplencia_),2)}%")

PJ inadimplentes
Quantidade: 35
Porcentagem: 19.02%


In [322]:
linhas = []

for profissao in profissoes:
    df_temp_profissao = df_inadimplencia_[df_inadimplencia_["Profissao"] == profissao].copy()
    df_temp_profissao = df_temp_profissao.reset_index(drop=True)
    total_registros = len(df_temp_profissao)

    contador_tipo = 0
    for indice_linha in range(len(df_temp_profissao)):
        registro_pj = df_temp_profissao.loc[indice_linha, "PJ"]
        if registro_pj:
            contador_tipo += 1

    linhas.append({
        "Profissao": profissao,
        "Total": total_registros,
        "Quant_PJ": contador_tipo,
        "%": round((contador_tipo * 100) / total_registros, 2),
    })

df_profissionais = pd.DataFrame(linhas)

for profissao in profissoes:
    print(df_profissionais[df_profissionais["Profissao"] == profissao])
    print()

  Profissao  Total  Quant_PJ      %
0  Dentista     34         7  20.59

  Profissao  Total  Quant_PJ      %
1    Outros     42        11  26.19

        Profissao  Total  Quant_PJ      %
2  Fisioterapeuta     33         5  15.15

    Profissao  Total  Quant_PJ      %
3  Enfermeiro     29         4  13.79

  Profissao  Total  Quant_PJ      %
4    Medico     46         8  17.39



---

### **Aqui fica comprovado que em 80% das profissões:**
- a maioria da inadimplência vem da solicitação do **Crédito Pessoal**, exceto em profissionais Médicos

In [323]:
linhas = []

for profissao in profissoes:
    df_temp_profissao = df_inadimplencia_[df_inadimplencia_["Profissao"] == profissao].copy()
    df_temp_profissao = df_temp_profissao.reset_index(drop=True)
    total_registros = len(df_temp_profissao)

    for tipo_contrato in lista_tipos:
        contador_tipo = 0
        for indice_linha in range(len(df_temp_profissao)):
            registro_tipo = df_temp_profissao.loc[indice_linha, "Tipo_Produto"]
            if tipo_contrato == registro_tipo:
                contador_tipo += 1

        linhas.append({
            "Profissao": profissao,
            "Tipo_Contrato": tipo_contrato,
            "Quant. Registros": contador_tipo,
            "%": round((contador_tipo * 100) / total_registros, 2),
        })

df_profissionais = pd.DataFrame(linhas)

for profissao in profissoes:
    print(df_profissionais[df_profissionais["Profissao"] == profissao])
    print()

  Profissao    Tipo_Contrato  Quant. Registros      %
0  Dentista  Credito Pessoal                14  41.18
1  Dentista     Equipamentos                13  38.24
2  Dentista  Capital de Giro                 7  20.59

  Profissao    Tipo_Contrato  Quant. Registros      %
3    Outros  Credito Pessoal                20  47.62
4    Outros     Equipamentos                11  26.19
5    Outros  Capital de Giro                11  26.19

        Profissao    Tipo_Contrato  Quant. Registros      %
6  Fisioterapeuta  Credito Pessoal                15  45.45
7  Fisioterapeuta     Equipamentos                13  39.39
8  Fisioterapeuta  Capital de Giro                 5  15.15

     Profissao    Tipo_Contrato  Quant. Registros      %
9   Enfermeiro  Credito Pessoal                19  65.52
10  Enfermeiro     Equipamentos                 6  20.69
11  Enfermeiro  Capital de Giro                 4  13.79

   Profissao    Tipo_Contrato  Quant. Registros      %
12    Medico  Credito Pessoal            

---